In [69]:
tabla='cbtci10'
dim='dim_tipcit'

In [70]:
import oracledb
import pandas as pd
from sqlalchemy import create_engine
import sys
from sqlalchemy import create_engine
from sqlalchemy import text
import sys
from decouple import config

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dl_essi"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena3  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine3 = create_engine(cadena3)
connection3 = engine3.connect()

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dw_essalud"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena4  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine4 = create_engine(cadena4)
connection4 = engine4.connect()

In [71]:


oracledb.init_oracle_client()
oracledb.version = "8.3.0"
sys.modules["cx_Oracle"] = oracledb
un = config("USER4_BDI_POSTGRES")
pw = config("PASS4_BDI_POSTGRES")
hostname=config("HOST4_BDI_POSTGRES")
service_name="WNET"
port = 1527

engine2 = create_engine(f'oracle://{un}:{pw}@',connect_args={
        "host": hostname,
        "port": port,
        "service_name": service_name
    }
)

connection2 = engine2.connect()

base2 = pd.read_sql_query(f"SELECT * FROM cbtci10", con=connection2)

connection2.close()




In [72]:
base2

,tipocitacod,tipocitanom,tipocitadescor
0,1,VOLUNTARIA,VOL
1,2,RECITA,REC
2,3,INTERCONSULTA,INT
3,4,REFERENCIA,REF
4,5,PROCEDIMIENTO,PRO
5,6,CONTROL,CTR


In [73]:
base2.columns

Index(['tipocitacod', 'tipocitanom', 'tipocitadescor'], dtype='object')

In [74]:

#################################################################################################################################################################################################################################################################################
#CREAMOS LA TABLA TEMPORAL Y LA SUBIMOS AL POSTGRES SQL

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dl_essi"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena3  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine3 = create_engine(cadena3)
connection3 = engine3.connect()


query_count_before = "SELECT COUNT(*) FROM cbtci10"
cant_antes = connection3.execute(query_count_before).fetchone()[0]
print(f"Cantidad de filas en la tabla cbtci10 antes de la inserción: {cant_antes}")


#connection3.execute('CREATE TEMPORARY TABLE tmp_cbtci10 ()')
base2.to_sql(name='tmp_cbtci10', con=engine3, if_exists='replace', index=False)

#################################################################################################################################################################################################################################################################################
#ACTUALIZAMOS EL cbtci10 FALSO CON LO OBTENIDO DEL ORACLE

query="""

ALTER TABLE tmp_cbtci10 
ALTER COLUMN tipocitanom  TYPE character(30),
ALTER COLUMN tipocitadescor  TYPE character(10),
ALTER COLUMN tipocitacod  TYPE character(1);

UPDATE cbtci10
SET 
tipocitanom = t.tipocitanom,
tipocitadescor = t.tipocitadescor,
tipocitacod = t.tipocitacod

FROM tmp_cbtci10 t 
WHERE cbtci10.tipocitacod = t.tipocitacod AND TRIM(cbtci10.tipocitacod) <> '' AND cbtci10.tipocitacod IS NOT NULL;

INSERT INTO cbtci10 
(tipocitanom,tipocitadescor,tipocitacod) 
SELECT 
tipocitanom,tipocitadescor,tipocitacod

FROM tmp_cbtci10  t
WHERE NOT EXISTS (
    SELECT 1 
    FROM cbtci10 
    WHERE cbtci10.tipocitacod = t.tipocitacod and cbtci10.tipocitacod IS NOT NULL
) ;


"""

c1= text(query)
connection3.execute(c1)

#BORRAMOS LAS TABLAS
query2="""
DROP TABLE tmp_cbtci10;
SELECT COUNT(*) FROM cbtci10;
"""


c2= text(query2)
cursor=connection3.execute(c2)
cant_despues = cursor.fetchone()[0]
print(f"Cantidad de filas en la tabla cbtci10 después de la inserción: {cant_despues}")
print(f"La cantidad de filas insertadas fue de: {cant_despues-cant_antes}")
base2 = pd.read_sql_query(f"SELECT * FROM cbtci10", con=connection3)


connection3.close()


Cantidad de filas en la tabla cbtci10 antes de la inserción: 6
Cantidad de filas en la tabla cbtci10 después de la inserción: 6
La cantidad de filas insertadas fue de: 0


In [75]:
#SUBIMOS ESA TABLA ACTUALIZADA COMO TEMPORAL A LA BASE DEL DIM ACTIVIT
base2.to_sql(name=f'tmp_{tabla}', con=engine4, if_exists='replace', index=False)

#ACTUALIZAMOS EL DIM ACTIVITI FALSO CON LA BASE DE DATOS QUE SUBIMOS
query_count_before = f"SELECT COUNT(*) FROM {dim}"
cant_antes = connection4.execute(query_count_before).fetchone()[0]
print(f"Cantidad de filas en la tabla {dim} antes de la inserción: {cant_antes}")

query=f"""

ALTER TABLE tmp_{tabla} 
ALTER COLUMN tipocitanom  TYPE character(30),
ALTER COLUMN tipocitadescor  TYPE character(10),
ALTER COLUMN tipocitacod  TYPE character(1);


INSERT INTO {dim} 

(des_tci,dco_tci,cod_tci) 

SELECT 

tipocitanom,tipocitadescor,tipocitacod

FROM tmp_{tabla} t
WHERE NOT EXISTS (
    SELECT 1 
    FROM {dim} d 
    WHERE 
    
    d.des_tci = t.tipocitanom AND
    d.dco_tci = t.tipocitadescor AND
    d.cod_tci = t.tipocitacod
);
"""



c1= text(query)
cursor=connection4.execute(c1)

#BORRAMOS LAS TABLAS
query2=f"""
DROP TABLE tmp_{tabla};
SELECT COUNT(*) FROM {dim};
"""

c2= text(query2)
cursor=connection4.execute(c2)
cant_despues = cursor.fetchone()[0]
print(f"Cantidad de filas en la tabla {dim} después de la inserción: {cant_despues}")
print(f"La cantidad de filas insertadas fue de: {cant_despues-cant_antes}")

Cantidad de filas en la tabla dim_tipcit antes de la inserción: 6
Cantidad de filas en la tabla dim_tipcit después de la inserción: 6
La cantidad de filas insertadas fue de: 0
